# Create National Heart Foundation of Australia Awards (GRANT PATTERN, annual recipient pages)

**Source:** the Heart Foundation's own "Research Grant Recipients {year}" pages,
`https://www.heartfoundation.org.au/research/research-award-recipients-{year}` (linked from
`/research/grant-recipients`), one accordion + recipient table per scheme. Rounds 2018-2025 are
online (earlier rounds are not). The 2024 tables are Canva embeds; the script reads the table
cells from the Canva view page JSON. Method 5 (static HTML). No bulk export exists; the
Heart Foundation is not a Europe PMC funder (GRIST has 0 hits).

**585 awards, rounds 2018-2025; 99% title, 100% lead researcher + institution,
27% amount (AUD; only the 2018-2019 pages publish amounts), 50% published grant number
(2022-2025).** Columns by era: 2018-19 amount + duration; 2020-21 project length; 2022-25
Award/Grant ID + research category. No dates are published: `start_year` = the funding-round
year of the page (projects commence in or after that year); `start_date`/`end_date` NULL.

**Excluded:** Innovation Awards / named awards (Shirley E Freeman, Paul Korner, Ross Hohnen:
honours to fellows already listed) and 4 "Honorary Future Leader Fellowship" rows (2021,
unfunded). Everything kept is research funding (fellowships, scholarships, Vanguard project
grants, First Nations / ATSI grants, collaboration & exchange travel grants).

**`funder_award_id` (§2.1.1):** citing works (`works?filter=grants.funder:F4320320442`,
`crossref_work.grants` stubs) carry the bare 6-digit Heart Foundation number (`106654`,
`102068`, `104751` ...), which is the numeric part of the page's Grant ID
(`110333-2025_FLF` -> `110333`); shipped for 2022-2025. 2018-2021 rows have no published
number: synthetic `NHFA-{year}-{sha1(scheme|name|title)[:10]}`. The full published Grant ID is
kept in the raw table (`grant_id_published`).

**Prerequisites:**
- Run `scripts/local/nhf_australia_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/nhf_australia/nhf_australia_projects.parquet` (§1.4 shrink guard).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320320442` (National Heart Foundation of Australia, AU). NOT the Heart
  Foundation of New Zealand (F4320320780) or the US "Heart Foundation" (F4320306962).
- display_name / ror_id / doi: read from `openalex.funders.funders` by ID (exactly one row)

**Priority:** `519` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).
**Provenance:** `nhf_australia_grant_recipients`.

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nhf_australia_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/nhf_australia/nhf_australia_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT funder_award_id) as distinct_ids FROM openalex.awards.nhf_australia_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.nhf_australia_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320320442 is a Crossref-registered funder, so it MUST resolve to exactly 1 row in
`openalex.funders.funders`. The assert fails the run otherwise.

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320320442 must resolve to exactly one openalex.funders.funders row')
FROM openalex.funders.funders WHERE funder_id = 4320320442;

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders WHERE funder_id = 4320320442;

## Step 2: Create Heart Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nhf_australia_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320320442  -- National Heart Foundation of Australia
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    -- 5 rows (2022 Collaboration and Exchange table) publish no project title
    COALESCE(NULLIF(TRIM(g.title), ''),
             CONCAT('Heart Foundation ', g.scheme, ' ', g.award_year)) as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'AUD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type as funding_type,
    NULLIF(TRIM(g.scheme), '') as funder_scheme,
    'nhf_australia_grant_recipients' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.award_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN g.lead_family_name IS NOT NULL OR g.lead_institution IS NOT NULL THEN
        named_struct(
            'given_name', NULLIF(TRIM(g.lead_given_name), ''),
            'family_name', NULLIF(TRIM(g.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.lead_institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        )
    END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.nhf_australia_raw g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- §2.1.1 shape check: bare 6-digit Heart Foundation number or the synthetic NHFA key; unique ids.
SELECT assert_true(
    SUM(CASE WHEN funder_award_id RLIKE '^([0-9]{6}|NHFA-20[0-9]{2}-[0-9a-f]{10})$' THEN 0 ELSE 1 END) = 0
    AND SUM(CASE WHEN funder_award_id LIKE '360G%' THEN 1 ELSE 0 END) = 0
    AND COUNT(*) = COUNT(DISTINCT id),
    'funder_award_id shape / uniqueness check failed')
FROM openalex.awards.nhf_australia_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nhf_australia_grant_recipients' AND priority = 519;

-- Priority 519: direct-from-funder ingest. Higher wins under the 2026-06-20
-- DESC dedup (oxjob #500), so it outranks the crossref/europepmc citation stubs
-- (priority 0/1) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    519 as priority
FROM openalex.awards.nhf_australia_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids FROM openalex.awards.nhf_australia_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.nhf_australia_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_aud
FROM openalex.awards.nhf_australia_awards GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, COUNT(amount) as with_amount, ROUND(SUM(amount), 0) as total_aud
FROM openalex.awards.nhf_australia_awards GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.family_name AS family, lead_investigator.given_name AS given, COUNT(*) AS n
FROM openalex.awards.nhf_australia_awards GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (amounts only published for the 2018-2019 rounds).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.nhf_australia_awards;

In [ ]:
%sql
-- Overlap with existing citation stubs for this funder (these collapse in CreateAwards dedup).
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(t.id) as matched_by_this_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.nhf_australia_awards t ON t.id = c.id
WHERE c.funder_id = 4320320442 AND c.priority < 519
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nhf_australia_grant_recipients'
GROUP BY provenance, priority;